<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TTS_Model_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TTS Model Loader — One-time Setup for the Whole TTS Suite

Pre-downloads all the model weights, code packages, and reference voice clips used by every notebook in this TTS lineup. Run **once**, then every other notebook in the suite starts instantly without re-downloading. 

### How It Works
1. Mounts your Google Drive and creates `/content/drive/MyDrive/AEI_TTS_Cache/` as the cache root
2. Downloads the selected model weights via `huggingface_hub.snapshot_download` to that folder
3. Installs the required Python packages (some via git+submodules)
4. Pre-caches bundled reference voice clips for the cloning tabs

All downloads are **resumable** \u2014 if the cell times out, just re-run; completed files won't be re-downloaded. Each notebook is independent in the checklist below, so you can pick just the ones you want to use.

### Total Storage
All 10 notebooks combined: **~35\u201340 GB** in Google Drive. Most users only need a few \u2014 pick below.

### Auth (only for `IndicF5`)
The `IndicF5` weights are gated on Hugging Face. To pre-cache them:
1. Create a read token at https://huggingface.co/settings/tokens
2. Accept the model terms at https://huggingface.co/ai4bharat/IndicF5
3. Add the token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar)

All other notebooks in the suite are ungated.

In [ ]:
# @title STEP 1 — Mount Drive and Choose What to Download
# @markdown Mounts your Google Drive, creates the cache root, and lets you pick which notebooks to pre-cache. The Drive is required so the cache survives between Colab sessions.

import os, sys, subprocess, importlib

# Drive mount (mandatory — cache lives in Drive so it persists across sessions)
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=False)
    print('[Drive] Mounted.')
else:
    print('[Drive] Already mounted.')

CACHE_ROOT = '/content/drive/MyDrive/AEI_TTS_Cache'
os.makedirs(CACHE_ROOT, exist_ok=True)
os.environ['HF_HOME'] = os.path.join(CACHE_ROOT, 'huggingface')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print(f'[Cache] {CACHE_ROOT}')
print(f'[Cache] HF_HOME={os.environ["HF_HOME"]}')

# Read HF_TOKEN for gated models
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
        print('[HF] HF_TOKEN loaded from Colab secrets.')
    else:
        print('[HF] HF_TOKEN not set (IndicF5 will be skipped).')
        HF_TOKEN = None
except Exception as e:
    print(f'[HF] Could not read secrets: {e} (IndicF5 will be skipped).')
    HF_TOKEN = None

# ============================================================
# CHECKLIST — toggle which notebooks to pre-cache
# ============================================================
DO_QWEN3_TTS  = True  # @param {type:"boolean"}  # Qwen3-TTS 1.7B — Apache 2.0, multilingual, 3 modes
DO_HIGGS      = True  # @param {type:"boolean"}  # Higgs Audio v3 (4B) — Research/Non-Commercial, 100+ langs
DO_MISO       = True  # @param {type:"boolean"}  # MisoTTS 8B — Other license, Sesame CSM, English
DO_SUPERTONIC = True  # @param {type:"boolean"}  # Supertonic-3 (99M) — OpenRAIL-M, 31 langs, ONNX
DO_DIA        = True  # @param {type:"boolean"}  # Dia 1.6B — Apache 2.0, non-verbals, English
DO_INDICF5    = False  # @param {type:"boolean"}  # IndicF5 (400M) — MIT, gated, 11 Indian languages
DO_MOSS       = True  # @param {type:"boolean"}  # MOSS-TTS v1.5 (8B) — Apache 2.0, 31 langs, IPA/Pinyin
DO_DOTS       = True  # @param {type:"boolean"}  # dots.tts-soar (2B) — Apache 2.0, SOTA multilingual SIM
DO_FISH       = True  # @param {type:"boolean"}  # Fish Audio S2 Pro (5B) — Research/Non-Commercial, 80+ langs
DO_VOICES     = True  # @param {type:"boolean"}  # Bundled reference voice clips for the cloning tabs

SELECTED = [n for n, on in dict(
    Qwen3_TTS=DO_QWEN3_TTS, Higgs=DO_HIGGS, MisoTTS=DO_MISO,
    Supertonic=DO_SUPERTONIC, Dia=DO_DIA, IndicF5=DO_INDICF5,
    MOSSTTS=DO_MOSS, dots_tts=DO_DOTS, FishS2=DO_FISH, Voices=DO_VOICES
).items() if on]
print(f'\n[Selection] {len(SELECTED)} item(s): {SELECTED}')

In [ ]:
# @title STEP 2 — Pre-cache All Selected Weights
# @markdown Downloads each selected model's weights to your Google Drive cache. Safe to re-run — completed files are skipped.

import os, time
from huggingface_hub import snapshot_download

# Each entry: (notebook, hf_repo, allow_patterns_or_None, size_estimate, gated)
MODELS = [
    ('Qwen3-TTS',   'Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice',        None, '~3.5 GB', False),
    ('Qwen3-TTS-VD','Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign',         None, '~3.5 GB', False),
    ('Qwen3-TTS-B', 'Qwen/Qwen3-TTS-12Hz-1.7B-Base',                None, '~3.5 GB', False),
    ('Higgs',       'multimodalart/higgs-audio-v3-tts-4b-transformers', None, '~10 GB', False),
    ('Higgs-codec', 'bosonai/higgs-audio-v2-tokenizer',             None, '~1 GB',  False),
    ('MisoTTS',     'MisoLabs/MisoTTS',                              None, '~30 GB', False),
    ('Supertonic',  'Supertone/supertonic-3',                       None, '~400 MB',False),
    ('Dia',         'nari-labs/Dia-1.6B-0626',                      None, '~3.2 GB',False),
    ('IndicF5',     'ai4bharat/IndicF5',                            None, '~800 MB',True),
    ('MOSS-TTS',    'OpenMOSS-Team/MOSS-TTS-v1.5',                  None, '~16 GB', False),
    ('dots.tts',    'rednote-hilab/dots.tts-soar',                  None, '~8 GB',  False),
    ('Fish-S2-Pro', 'fishaudio/s2-pro',                             None, '~10 GB', False),
]
MODEL_BY_NAME = {n: (r, sz, g) for (n, r, _, sz, g) in MODELS}

# Map notebook selection \u2192 which model rows to download
NOTEBOOK_DEPS = {
    'Qwen3_TTS': ['Qwen3-TTS', 'Qwen3-TTS-VD', 'Qwen3-TTS-B'],
    'Higgs':      ['Higgs', 'Higgs-codec'],
    'MisoTTS':    ['MisoTTS'],
    'Supertonic': ['Supertonic'],
    'Dia':        ['Dia'],
    'IndicF5':    ['IndicF5'],
    'MOSSTTS':    ['MOSS-TTS'],
    'dots_tts':   ['dots.tts'],
    'FishS2':     ['Fish-S2-Pro'],
}

# Build the queue of (display_name, repo, size, gated)
queue = []
seen = set()
for nb in SELECTED:
    if nb == 'Voices':
        continue
    for model_name in NOTEBOOK_DEPS.get(nb, []):
        if model_name in seen:
            continue
        seen.add(model_name)
        repo, size, gated = MODEL_BY_NAME[model_name]
        queue.append((model_name, repo, size, gated))

if not queue:
    print('[Cache] Nothing selected. Skipping.')
else:
    print(f'[Cache] {len(queue)} model(s) to download:')
    for n, r, s, g in queue:
        tag = ' (gated, needs HF_TOKEN)' if g else ''
        print(f'  \u00b7 {n}  {s}{tag}  <-  {r}')
    print()

    failed = []
    for name, repo, size, gated in queue:
        if gated and not HF_TOKEN:
            print(f'[Skip] {name} (gated, no HF_TOKEN) — set the secret and re-run.')
            continue
        print(f'[Download] {name}  {repo}  ({size}) ...')
        t0 = time.time()
        try:
            snapshot_download(repo_id=repo, token=HF_TOKEN, allow_patterns=None)
            print(f'[Done] {name} in {time.time()-t0:.0f}s\n')
        except Exception as e:
            print(f'[Fail] {name}: {e}\n')
            failed.append((name, str(e)))

    if failed:
        print('\n[Summary] FAILED:')
        for n, e in failed:
            print(f'  \u274c {n}: {e[:200]}')
    else:
        print('\n[Summary] All selected models cached successfully.')

In [ ]:
# @title STEP 3 — Install Required Python Packages
# @markdown Installs the code packages each notebook depends on. Skip the ones you don't need by leaving the toggle off in Step 1.

import subprocess, sys, importlib

PKG_PLAN = [
    ('Qwen3_TTS',  ['qwen-tts'],                                                                     'pip'),
    ('Higgs',      [],                                                                               'transformers>=5.5 already pulled in by qwen-tts'),
    ('MisoTTS',    ['moshi==0.2.2', 'silentcipher @ git+https://github.com/SesameAILabs/silentcipher@d46d7d0893a583d8968ab3a6626e2289faec9152'], 'pip'),
    ('Supertonic', ['supertonic'],                                                                   'pip'),
    ('Dia',        ['git+https://github.com/nari-labs/dia.git'],                                     'pip'),
    ('IndicF5',    ['git+https://github.com/ai4bharat/IndicF5.git'],                                 'pip'),
    ('MOSSTTS',    ['git+https://github.com/OpenMOSS/MOSS-TTS.git'],                                 'pip'),
    ('dots_tts',   ['git+https://github.com/rednote-hilab/dots.tts.git'],                            'pip'),
    ('FishS2',     ['git+https://github.com/fishaudio/fish-speech.git'],                              'pip'),
    # Voice clips are pure downloads, no Python packages
]

for nb, pkgs, kind in PKG_PLAN:
    if nb not in SELECTED:
        continue
    if not pkgs:
        print(f'[Skip] {nb}: no extra packages ({kind})')
        continue
    print(f'[pip] {nb}: installing {len(pkgs)} package(s) ...')
    cmd = ['pip', 'install', '-q', '-U'] + pkgs
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        print(f'  \u2705 installed')
    else:
        print(f'  \u26a0\ufe0f install had warnings (rc={res.returncode}); last 300 chars of stderr:')
        print(f'  {res.stderr[-300:]}')
print('\n[Done] Package install step complete.')

In [ ]:
# @title STEP 4 — Pre-cache Bundled Reference Voice Clips
# @markdown Downloads the example reference audio clips that ship with each notebook, so cloning tabs have instant demos on first launch.

import os, subprocess, urllib.request

VOICE_DIR = os.path.join(CACHE_ROOT, 'voices')
os.makedirs(VOICE_DIR, exist_ok=True)

# Each entry: (subdir, filename, source_url)
VOICES = [
    # IndicF5 — 7 prompts from the GitHub repo
    ('IndicF5', 'KAN_F_HAPPY_00001.wav',  'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/KAN_F_HAPPY_00001.wav'),
    ('IndicF5', 'MAR_F_HAPPY_00001.wav',  'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/MAR_F_HAPPY_00001.wav'),
    ('IndicF5', 'MAR_F_WIKI_00001.wav',   'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/MAR_F_WIKI_00001.wav'),
    ('IndicF5', 'MAR_M_WIKI_00001.wav',   'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/MAR_M_WIKI_00001.wav'),
    ('IndicF5', 'PAN_F_HAPPY_00001.wav',  'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/PAN_F_HAPPY_00001.wav'),
    ('IndicF5', 'PAN_F_HAPPY_00002.wav',  'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/PAN_F_HAPPY_00002.wav'),
    ('IndicF5', 'TAM_F_HAPPY_00001.wav',  'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/TAM_F_HAPPY_00001.wav'),
    # Dia — example_prompt.mp3 ships with the repo
    ('Dia',     'example_prompt.mp3',     'https://github.com/nari-labs/dia/raw/main/example_prompt.mp3'),
]

if 'Voices' not in SELECTED:
    print('[Skip] Voices toggle is off in Step 1.')
else:
    print(f'[Voices] Downloading to {VOICE_DIR} ...')
    for subdir, fname, url in VOICES:
        out_dir = os.path.join(VOICE_DIR, subdir)
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, fname)
        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            print(f'  \u2705 {subdir}/{fname} (cached)')
            continue
        try:
            urllib.request.urlretrieve(url, out_path)
            sz = os.path.getsize(out_path)
            print(f'  \u2705 {subdir}/{fname} ({sz//1024} KB)')
        except Exception as e:
            print(f'  \u274c {subdir}/{fname}: {e}')
    print('\n[Voices] Done.')

In [ ]:
# @title STEP 5 — Summary and Next Steps
# @markdown Shows the final cache state on Google Drive and tells you what to do next.

import os, subprocess

def dir_size_mb(p):
    total = 0
    for root, _, files in os.walk(p):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return total / 1024**2

print(f'\n[Cache] {CACHE_ROOT}')
print(f'  huggingface/: {dir_size_mb(os.path.join(CACHE_ROOT, "huggingface")):.0f} MB')
print(f'  voices/:      {dir_size_mb(os.path.join(CACHE_ROOT, "voices")):.0f} MB')
print()
print('[Next steps]')
print('  1. Open any of the TTS notebooks in this repo. They will skip the weight download step')
print('     and load directly from the Drive cache.')
print('  2. If you skipped a notebook here and want to add it later, just re-run this loader with')
print('     the new toggle enabled. The Drive cache is the source of truth.')
print('  3. To free space, delete subfolders from /content/drive/MyDrive/AEI_TTS_Cache/huggingface/')
print('     manually — or wipe the whole AEI_TTS_Cache folder to start fresh.')